# TLE history — orbital elements with Orekit

Reads the TLE history of one object (file `<NORAD>.txt` written by `spacetrack_tle_history.py`
in the folder set in `config.ini`, section `[tle_history]`), parses each TLE with Orekit and plots
the orbital elements over time:

- **Mean elements** (SGP4/TEME), read directly from each TLE by Orekit.
- **Osculating Keplerian elements** (EME2000), from the position/velocity propagated by
  Orekit's `TLEPropagator` at each TLE epoch.

The Orekit data file is set in `config.ini`, section `[orekit]`, key `data_path`.

In [ ]:
from pathlib import Path
import configparser
import math

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

NORAD_ID = 8820
START    = None   # e.g. "2020-01-01" (None: from the first TLE)
END      = None   # e.g. "2024-12-31" (None: up to the last TLE)

# Folders read from config.ini (same file as spacetrack_tle_history.py)
config = configparser.ConfigParser()
if not config.read("config.ini", encoding="utf-8"):
    raise FileNotFoundError("config.ini not found next to the notebook")
HISTORY_DIR = Path(config["tle_history"]["output_dir"]).expanduser()
TLE_FILE    = HISTORY_DIR / f"{NORAD_ID}.txt"
OREKIT_DATA = Path(config.get("orekit", "data_path", fallback="orekit-data.zip")).expanduser()

R_EARTH = 6_378.137   # km — WGS84 equatorial radius

## Start Orekit

In [ ]:
import orekit
orekit.initVM()

from orekit.pyhelpers import setup_orekit_curdir, absolutedate_to_datetime
if not OREKIT_DATA.exists():
    raise FileNotFoundError(f"Orekit data not found: {OREKIT_DATA} (see [orekit] data_path in config.ini)")
setup_orekit_curdir(str(OREKIT_DATA))

from org.orekit.frames import FramesFactory
from org.orekit.orbits import KeplerianOrbit
from org.orekit.propagation.analytical.tle import TLE, TLEPropagator
from org.orekit.utils import Constants

EME2000 = FramesFactory.getEME2000()
MU      = Constants.WGS84_EARTH_MU   # m³/s²
print(f"Orekit {orekit.VERSION} ready — data: {OREKIT_DATA}")

## Read the TLE file

In [ ]:
def read_tle_history(path: Path) -> list[tuple[str, str]]:
    """Read a 2-line TLE file and return the list of (line1, line2) pairs."""
    lines = [l.strip() for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    pairs = []
    i = 0
    while i < len(lines) - 1:
        line1, line2 = lines[i], lines[i + 1]
        if line1.startswith("1 ") and line2.startswith("2 "):
            pairs.append((line1, line2))
            i += 2
        else:
            i += 1  # isolated / malformed line (e.g. object name) -> skip it
    return pairs


pairs = read_tle_history(TLE_FILE)
print(f"{len(pairs):,} TLEs read from {TLE_FILE}")

## Extract the orbital elements with Orekit

For each TLE: the mean elements come from the `TLE` object (angles in radians, mean motion in rad/s;
the semi-major axis is derived from the mean motion), and the osculating elements from a
`KeplerianOrbit` built on the state propagated at the TLE epoch.

In [ ]:
def tle_elements(line1: str, line2: str) -> dict:
    tle  = TLE(line1, line2)
    date = tle.getDate()

    n   = tle.getMeanMotion()                 # rad/s
    sma = (MU / n**2) ** (1 / 3) / 1000       # km
    ecc = tle.getE()
    record = {
        "epoch":       absolutedate_to_datetime(date),
        "sma":         sma,
        "ecc":         ecc,
        "inc":         math.degrees(tle.getI()),
        "raan":        math.degrees(tle.getRaan()),
        "aop":         math.degrees(tle.getPerigeeArgument()),
        "ma":          math.degrees(tle.getMeanAnomaly()),
        "mean_motion": n * 86_400 / (2 * math.pi),   # rev/day
        "bstar":       tle.getBStar(),
        "perigee_alt": sma * (1 - ecc) - R_EARTH,
        "apogee_alt":  sma * (1 + ecc) - R_EARTH,
    }

    # Osculating Keplerian elements in EME2000 at the TLE epoch
    pv  = TLEPropagator.selectExtrapolator(tle).propagate(date).getPVCoordinates(EME2000)
    osc = KeplerianOrbit(pv, EME2000, date, MU)
    record.update({
        "osc_sma":  osc.getA() / 1000,
        "osc_ecc":  osc.getE(),
        "osc_inc":  math.degrees(osc.getI()),
        "osc_raan": math.degrees(osc.getRightAscensionOfAscendingNode()) % 360,
        "osc_aop":  math.degrees(osc.getPerigeeArgument()) % 360,
        "osc_ma":   math.degrees(osc.getMeanAnomaly()) % 360,
    })
    return record


records, n_failed = [], 0
for i, (line1, line2) in enumerate(pairs, 1):
    try:
        records.append(tle_elements(line1, line2))
    except Exception as exc:
        n_failed += 1
        if n_failed <= 5:
            print(f"  [skip] TLE #{i}: {exc}")
    if i % 2000 == 0 or i == len(pairs):
        print(f"  {i:,}/{len(pairs):,} TLEs processed")

df = pd.DataFrame(records).sort_values("epoch").drop_duplicates("epoch").reset_index(drop=True)
if START:
    df = df[df["epoch"] >= pd.Timestamp(START)]
if END:
    df = df[df["epoch"] <= pd.Timestamp(END)]
df = df.reset_index(drop=True)

print(f"{len(df):,} TLEs kept ({n_failed} rejected by Orekit): {df['epoch'].min()} → {df['epoch'].max()}")
df.head()

## Plot

In [ ]:
def plot_elements(df: pd.DataFrame, elements: list[tuple[str, str, str]], title: str) -> None:
    fig, axes = plt.subplots(len(elements), 1, figsize=(12, 2.2 * len(elements)), sharex=True)
    fig.patch.set_facecolor("#1a1a2e")

    for ax, (col, label, color) in zip(axes, elements):
        ax.set_facecolor("#16213e")
        # Angles wrapping at 360° are drawn as points only, to avoid vertical jump lines
        wrapped = col.removeprefix("osc_") in ("raan", "aop", "ma")
        ax.plot(df["epoch"], df[col], color=color, linewidth=0.9, marker=".", markersize=2,
                linestyle="none" if wrapped else "-")
        ax.set_ylabel(label, color="white", fontsize=8)
        ax.tick_params(colors="white", labelsize=7)
        ax.yaxis.set_label_coords(-0.08, 0.5)
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")
        ax.grid(axis="y", color="#333", linewidth=0.5)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
    fig.autofmt_xdate(rotation=30, ha="right")
    axes[-1].tick_params(axis="x", colors="white", labelsize=7)

    fig.suptitle(
        f"{title} — NORAD {NORAD_ID}  "
        f"({df['epoch'].min().date()} → {df['epoch'].max().date()})\n{len(df):,} TLEs",
        color="white", fontsize=12, y=0.995,
    )
    fig.tight_layout()
    plt.show()

### Mean elements (TLE, SGP4/TEME)

In [ ]:
plot_elements(df, [
    ("sma",         "Semi-major axis (km)",      "#64B5F6"),
    ("ecc",         "Eccentricity",              "#FFB74D"),
    ("inc",         "Inclination (°)",           "#81C784"),
    ("raan",        "RAAN (°)",                  "#CE93D8"),
    ("aop",         "Argument of perigee (°)",   "#F48FB1"),
    ("ma",          "Mean anomaly (°)",          "#80DEEA"),
    ("perigee_alt", "Perigee altitude (km)",     "#AED581"),
    ("apogee_alt",  "Apogee altitude (km)",      "#FF8A65"),
], "Mean orbital elements")

### Osculating Keplerian elements (EME2000)

In [ ]:
plot_elements(df, [
    ("osc_sma",  "Semi-major axis (km)",    "#64B5F6"),
    ("osc_ecc",  "Eccentricity",            "#FFB74D"),
    ("osc_inc",  "Inclination (°)",         "#81C784"),
    ("osc_raan", "RAAN (°)",                "#CE93D8"),
    ("osc_aop",  "Argument of perigee (°)", "#F48FB1"),
    ("osc_ma",   "Mean anomaly (°)",        "#80DEEA"),
], "Osculating orbital elements")